# One-sample correlations: which gene pairs are co-expressed?

This tutorial estimates gene-gene correlations within a single population of cells and tests each one against zero. A typical use is finding the genes whose expression tracks a transcription factor in one cell type.

**What the one-sample test does**

- It runs when `ht_2d_moments` is called without a treatment (`treatment=None`).
- The estimate is the average of the per-group correlations, weighted by each group's cell count. Here every donor is a group.
- The null hypothesis is that the correlation is zero.
- It takes no covariates. A one-sample average has nothing to adjust: to describe a particular population (one condition, one cell type, healthy donors only), subset the data to that population first.

**Two things to keep in mind when reading the results**

- Because groups are weighted by cell count, a donor with many cells has more influence on the estimate.
- The p-value reflects the uncertainty from sampling cells within each donor. It does not reflect variation between donors, so a pair can be significant because of one or two donors. The last section shows how to check this.

To compare correlations between conditions instead, pass a treatment; see the differential correlation guide in the documentation.

Data: https://memento-examples.s3.us-west-2.amazonaws.com/pbmc-ifnb/interferon_filtered.h5ad (PBMCs from 8 donors, with and without interferon-β).

In [ ]:
import itertools

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc

import memento

In [ ]:
data_path = './'  # folder containing interferon_filtered.h5ad

### Read the data and select the population

`adata.X` must hold raw integer counts in a sparse matrix, ideally with all detected genes.

The one-sample test describes whatever cells you give it, so select the population here: unstimulated CD14+ monocytes.

In [ ]:
adata = sc.read(data_path + 'interferon_filtered.h5ad')
adata = adata[(adata.obs['cell'] == 'CD14+ Monocytes') & (adata.obs['stim'] == 'ctrl')].copy()
adata.obs['ind'] = adata.obs['ind'].astype(str)
print(adata.shape)
adata.obs['ind'].value_counts()

### Set up memento and group cells by donor

`capture_rate` is the overall UMI capture efficiency (0.07 for this 10x v1 dataset). Each donor (`ind`) becomes one group.

`min_perc_group=.9` keeps genes that pass the expression filter in at least 90% of donors.

In [ ]:
adata.obs['capture_rate'] = 0.07
memento.setup_memento(adata, q_column='capture_rate')
memento.create_groups(adata, label_columns=['ind'])
memento.compute_1d_moments(adata, min_perc_group=.9)
print(adata.shape[1], 'genes retained')

### Choose the gene pairs

Pair each transcription factor with every other retained gene. Only genes that survived filtering can be tested.

Each pair is bootstrapped separately, so runtime grows with the number of pairs. For long lists of transcription factors, run them in batches.

In [ ]:
tfs = [tf for tf in ['STAT1', 'IRF7'] if tf in adata.var_names]
gene_pairs = [(tf, gene) for tf, gene in itertools.product(tfs, adata.var_names) if tf != gene]
print(tfs, len(gene_pairs), 'pairs')

### Run the one-sample test

Leaving out `treatment` requests the one-sample test. Increase `num_cpus` to match your machine.

In [ ]:
memento.compute_2d_moments(adata, gene_pairs)
memento.ht_2d_moments(
    adata,
    treatment=None,
    num_boot=5000,
    num_cpus=4,
    verbose=1)
result = memento.get_2d_ht_result(adata)
result.head()

`corr_coef` is the cell-count-weighted mean correlation across donors, `corr_se` its bootstrap standard error, and `corr_pval` the unadjusted p-value against zero. The `tx` column reads `intercept` for one-sample tests.

P-values are not corrected for multiple testing. Apply a correction within each transcription factor:

In [ ]:
result['corr_fdr'] = result.groupby('gene_1')['corr_pval'].transform(memento.util._fdrcorrect)
(result['corr_fdr'] < 0.1).groupby(result['gene_1']).sum()

### Check that donors agree

`get_2d_moments` returns the correlation estimated within each donor. Use it to see whether a significant pair is supported across donors or carried by a few of them.

Below, `donor_agreement` is the fraction of donors whose correlation has the same sign as the overall estimate. Donors where the correlation could not be estimated are left out of the fraction.

In [ ]:
per_donor = memento.get_2d_moments(adata, groupby='ind').set_index(['gene_1', 'gene_2'])
per_donor.head()

In [ ]:
result = result.set_index(['gene_1', 'gene_2'])
same_sign = np.sign(per_donor).eq(np.sign(result['corr_coef']), axis=0)
result['n_donors'] = per_donor.notna().sum(axis=1)
result['donor_agreement'] = same_sign.sum(axis=1) / result['n_donors']
result = result.reset_index()

supported = result.query('corr_fdr < 0.1 & donor_agreement >= 0.75')
print(len(result.query('corr_fdr < 0.1')), 'pairs at FDR < 0.1;', len(supported), 'of them agree in sign in at least 75% of donors')
supported.sort_values('corr_pval').head(20)

### Plot

Grey points are all tested pairs; coloured points pass FDR < 0.1 and agree in sign in at least 75% of donors.

In [ ]:
fig, axes = plt.subplots(1, len(tfs), figsize=(4 * len(tfs), 3), sharey=True, squeeze=False)
for ax, tf in zip(axes[0], tfs):
    df = result[result['gene_1'] == tf]
    hits = supported[supported['gene_1'] == tf]
    ax.scatter(df['corr_coef'], -np.log10(df['corr_pval']), s=1, color='grey')
    ax.scatter(hits['corr_coef'], -np.log10(hits['corr_pval']), s=4)
    ax.set_title(tf)
    ax.set_xlabel('correlation')
axes[0][0].set_ylabel('-log10 p-value')
plt.tight_layout()

### Notes

- **Choosing the population.** Covariates such as disease status or condition are handled by subsetting before `create_groups`, as done here for unstimulated cells. Repeat the analysis on another subset to describe another population.
- **Comparing populations.** To test whether a correlation differs between two conditions, keep both in the data and pass the condition as `treatment` to `ht_2d_moments`. Covariates are used in that test.
- **Few donors.** With a small number of donors, treat the significance of any single pair with care and rely on the per-donor table.